# SeqTrainer Tutorial 09: Gemma 3 4B for promoter classification + regression

This notebook follows the same end-to-end style as prior tutorials while using a long-context Gemma 3 4B backbone through SeqTrainer package helpers.


## Goal

- Use `seqtrainer.torch.gemma3_4b(...)` for backbone selection.
- Run promoter **classification** and **regression** with a shared encoder and simple task heads.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

from seqtrainer.torch import build_finetune_config, gemma3_4b

print('Imports loaded')


## 1) Load promoter data

This example uses the packaged dataset builder output and creates fallback labels when needed.


In [ ]:
df = pd.read_csv('../data/dataset_builder/processed_dataset.csv')

if 'sequence' not in df.columns:
    seq_col = [c for c in df.columns if 'sequence' in c.lower()][0]
    df = df.rename(columns={seq_col: 'sequence'})

if 'activity' not in df.columns:
    target_col = 'expression' if 'expression' in df.columns else df.columns[-1]
    df['activity'] = df[target_col].astype(float)

if 'label' not in df.columns:
    threshold = df['activity'].median()
    df['label'] = (df['activity'] >= threshold).astype(int)

df = df[['sequence', 'label', 'activity']].dropna().reset_index(drop=True)
print(f'Rows: {len(df)}')
df.head()


## 2) Configure Gemma 3 4B backbone with SeqTrainer


In [ ]:
backbone = gemma3_4b(instruct_tuned=True)
cfg = build_finetune_config(
    backbone=backbone.model_id,
    head='classification_or_regression',
    learning_rate=2e-5,
    epochs=3,
)

print(backbone)
print(cfg)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(backbone.tokenizer_id, trust_remote_code=backbone.trust_remote_code)
encoder = AutoModel.from_pretrained(backbone.model_id, trust_remote_code=backbone.trust_remote_code).to(device)

print('Device:', device)


## 3) Datasets, collator, and heads


In [ ]:
class PromoterDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, target_col: str):
        self.frame = frame.reset_index(drop=True)
        self.target_col = target_col

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        return row['sequence'], float(row[self.target_col])


def collate_batch(batch):
    seqs, ys = zip(*batch)
    tok = tokenizer(
        list(seqs),
        padding=True,
        truncation=True,
        max_length=min(512, backbone.max_length),
        return_tensors='pt',
    )
    return tok, torch.tensor(ys, dtype=torch.float32)


class MLPHead(nn.Module):
    def __init__(self, hidden_size: int):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x):
        return self.layers(x).squeeze(-1)


## 4) Train/eval helpers


In [ ]:
def masked_mean_pool(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).float()
    summed = (hidden_states * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp_min(1e-6)
    return summed / denom


def forward_encoder(batch_tokens):
    batch_tokens = {k: v.to(device) for k, v in batch_tokens.items()}
    out = encoder(**batch_tokens)
    return masked_mean_pool(out.last_hidden_state, batch_tokens['attention_mask'])


def train_epoch(head, loader, optimizer, loss_fn):
    encoder.train()
    head.train()
    losses = []
    for tokens, y in loader:
        y = y.to(device)
        optimizer.zero_grad()
        z = forward_encoder(tokens)
        pred = head(z)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def infer(head, loader):
    encoder.eval()
    head.eval()
    preds, ys = [], []
    for tokens, y in loader:
        y = y.to(device)
        z = forward_encoder(tokens)
        pred = head(z)
        preds.extend(pred.cpu().numpy().tolist())
        ys.extend(y.cpu().numpy().tolist())
    return np.array(preds), np.array(ys)


## 5) Promoter classification


In [ ]:
tr_df, te_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_loader = DataLoader(PromoterDataset(tr_df, 'label'), batch_size=8, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(PromoterDataset(te_df, 'label'), batch_size=8, shuffle=False, collate_fn=collate_batch)

head = MLPHead(encoder.config.hidden_size).to(device)
opt = torch.optim.AdamW(list(encoder.parameters()) + list(head.parameters()), lr=cfg['learning_rate'])
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(cfg['epochs']):
    loss = train_epoch(head, train_loader, opt, loss_fn)
    print(f'classification epoch {epoch+1}: {loss:.4f}')

logits, y_true = infer(head, test_loader)
probs = 1 / (1 + np.exp(-logits))
y_pred = (probs >= 0.5).astype(int)

print('Accuracy:', accuracy_score(y_true, y_pred))
print('F1:', f1_score(y_true, y_pred))


## 6) Promoter regression


In [ ]:
tr_df, te_df = train_test_split(df, test_size=0.2, random_state=42)

train_loader = DataLoader(PromoterDataset(tr_df, 'activity'), batch_size=8, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(PromoterDataset(te_df, 'activity'), batch_size=8, shuffle=False, collate_fn=collate_batch)

head = MLPHead(encoder.config.hidden_size).to(device)
opt = torch.optim.AdamW(list(encoder.parameters()) + list(head.parameters()), lr=cfg['learning_rate'])
loss_fn = nn.MSELoss()

for epoch in range(cfg['epochs']):
    loss = train_epoch(head, train_loader, opt, loss_fn)
    print(f'regression epoch {epoch+1}: {loss:.4f}')

pred, y_true = infer(head, test_loader)
print('RMSE:', mean_squared_error(y_true, pred, squared=False))
print('R2:', r2_score(y_true, pred))
